# Item 4 — Data collection exercise

**Problem.** My research capsule contains a passive magneto-mechanical resonator (MMR). Its position is
computed from coil signal amplitudes, and it must be re-excited at its resonance, so every short
receive window (16–50 ms) has to yield the resonator's **frequency, amplitude and phase**. My course
project asks whether a learned estimator can do this better than FFT/least-squares fits. Before relying
on simulation, I want a small batch of **real** MMR recordings to check what real windows look like.

**Source.** I do not have my own measurements yet, so I use the open dataset of
Reiss et al., *Parameter Estimation for Model-Based Sensing of Magneto-Mechanical Resonators*,
Communications Physics (2026); data DOI [10.15480/882.16742](https://doi.org/10.15480/882.16742),
license CC0 1.0. I use the smaller sensor (**MMR S**, Experiment 1, uncontrolled), because its
frequency band (600–2000 Hz in the authors' filter settings) is closest to my design (≈570 Hz).

## Step 1 — Table sketch (one row = one short window of one frame of one receive channel)

| column | type | role | meaning |
|---|---|---|---|
| `frame` | int | id | measurement frame index in the raw file |
| `channel` | int | id | receive coil channel |
| `t_start_s` | float | id | window start inside the frame [s] |
| `n_samples`, `fs_hz` | int, float | meta | window length, sampling rate |
| `x[0..N-1]` | float32 array | **feature** | raw (band-passed) samples of the window — stored in `X.npy`, row-aligned |
| `rel_amplitude`, `snr_db_est` | float | meta | how strong the window is (used for filtering/stratifying) |
| `f_ref_hz` | float | **target** | resonance frequency |
| `A_ref` | float | **target** | amplitude at window (arbitrary units — see open questions) |
| `phi_ref_rad` | float | **target** | phase at the window's first sample |
| `ref_residual_rel` | float | quality | residual of the reference fit / signal RMS |

**Where do targets come from?** Real data has no ground truth. I use a *pseudo-label*: a sinusoid fit on
a **4× longer** segment (200 ms) centred on each 50 ms window. A longer segment gives a much lower-variance
estimate, so it can act as the reference for the short window. Caveat: if the frequency drifts within
200 ms, the label is biased (open question below).

In [ ]:
import os, sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

# Repo root = folder that contains raw/, processed/, scripts/  (adjust if your notebook lives elsewhere)
ROOT = Path("mmr-ml")
sys.path.insert(0, str(ROOT / "scripts"))
import mmr_data as md

RAW = Path(os.environ.get("MMR_RAW", ROOT / "raw" / "Exp1_MMRS.h5"))
PROC = ROOT / "processed"; PROC.mkdir(exist_ok=True)
assert RAW.exists(), f"Download Exp1_MMRS.h5 from https://doi.org/10.15480/882.16742 into {RAW.parent}"
print("raw file:", RAW, f"({RAW.stat().st_size/1e6:.1f} MB)")

## Step 2 — Load the first batch

First I look at the file structure instead of trusting assumptions: the authors' Julia loader reads
`/data` and `/sequence/{baseFrequency, decimation}`, with sampling rate = baseFrequency / decimation.
Julia stores arrays column-major, so Python may see the axes reversed — `load_raw` detects the time axis
as the longest one and reorders to *(frame, channel, sample)*.

In [ ]:
raw_sha = md.sha256(RAW)                       # fingerprint: proves which file the results came from
print("sha256:", raw_sha)
print("\n".join(md.describe_h5(RAW)))

data, fs, load_info = md.load_raw(RAW)
F, Cn, S = data.shape
print(load_info)
print(f"{F} frames x {Cn} channels x {S} samples  ->  {S/fs:.3f} s per frame at fs = {fs:.1f} Hz")

## Step 3 — Cleanup (four kinds of dirty data)

Each check prints what it found; every action is recorded in `cleaning_log`, which is saved to
`metadata.yaml`. Order: format first (everything else needs the right array layout), then incorrect
values (filtering), then incomplete records, then irrelevant/duplicated ones.

In [ ]:
cleaning_log = []

# --- IMPROPER FORMAT -------------------------------------------------------------------
# * Julia (column-major) layout -> axes reordered to (frame, channel, sample) in load_raw
# * stored as float32 -> converted to float64 for the fits
# * sampling rate is not a column but two fields (baseFrequency / decimation) -> fs computed
# * feature table is "wide": one window = 3052 samples -> kept in X_windows.npy, row-aligned
#   with the CSV, instead of 3052 CSV columns
t = np.arange(S) / fs
fr0 = 0
fig, ax = plt.subplots(Cn, 1, figsize=(10, 1.8 * Cn), sharex=True, squeeze=False)
for c in range(Cn):
    ax[c, 0].plot(t, data[fr0, c], lw=0.5); ax[c, 0].set_ylabel(f"ch {c}")
ax[-1, 0].set_xlabel("t [s]"); fig.suptitle(f"raw frame {fr0}"); plt.tight_layout(); plt.show()
cleaning_log.append({"kind": "improper format", "stored": load_info["stored_shape"],
                     "action": "reordered to (frame, channel, sample), float64, fs = baseFrequency/decimation; "
                               "window samples stored in X_windows.npy"})

In [ ]:
# --- INCORRECT -------------------------------------------------------------------------
# Values that are not the resonator's signal:
# * clipping (ADC saturation) would give impossible flat tops -> flagged
# * DC offset, mains (50 Hz) harmonics below 600 Hz, broadband noise -> band-pass filter
# * filter transients and the jump in the last samples of each frame -> edges cut
clip = md.clipped_fraction(np.nan_to_num(data))                 # (frame, channel)
clip_frames = sorted(set(np.where((clip > 0.001).any(axis=1))[0].tolist()))
print("frames with possible clipping (>0.1 % samples at max):", clip_frames)

BAND = (600.0, 2000.0)                           # authors' pass-band for MMR S (example.jl)
clean = md.bandpass(np.nan_to_num(data), fs, BAND)   # zero-phase -> phase is not shifted
edge = int(0.05 * fs)                            # 50 ms at both ends
clean = clean[..., edge:S - edge]

f_ax = np.fft.rfftfreq(S, 1 / fs)
fig, ax = plt.subplots(figsize=(10, 3))
ax.semilogy(f_ax, np.abs(np.fft.rfft(np.nan_to_num(data[fr0, 0]))), lw=0.5, label="raw")
ax.semilogy(np.fft.rfftfreq(clean.shape[-1], 1 / fs), np.abs(np.fft.rfft(clean[fr0, 0])), lw=0.5, label="band-passed")
ax.set_xlim(0, 2500); ax.set_xlabel("f [Hz]"); ax.legend(); plt.show()
cleaning_log.append({"kind": "incorrect", "clipping_frames": clip_frames,
                     "action": f"clipping flagged only; 4th-order Butterworth {BAND} Hz (filtfilt); "
                               f"cut {edge} samples per edge. Odd 50 Hz harmonics inside the band remain."})

In [ ]:
# --- INCOMPLETE ------------------------------------------------------------------------
# * missing samples (NaN/Inf) inside a frame -> frame dropped (a gap breaks the sinusoid fit)
# * "sensor dropped out": a frame whose signal is far weaker than usual on every channel
# * empty metadata fields: units of /data are undocumented -> listed as an open question
nan_frames = sorted(set(np.where((~np.isfinite(data)).any(axis=(1, 2)))[0].tolist()))
rms = np.sqrt(np.mean(clean ** 2, axis=-1))                     # (frame, channel)
dead_frames = np.where((rms < 0.1 * np.median(rms, axis=0)).all(axis=1))[0].tolist()
print("frames with NaN/Inf:", nan_frames, "| dead frames:", dead_frames)
cleaning_log.append({"kind": "incomplete", "nan_frames": nan_frames, "dead_frames": dead_frames,
                     "action": "drop frames with NaN/Inf or with RMS < 10 % of median on all channels; "
                               "units of /data marked UNKNOWN in metadata"})

In [ ]:
# --- IRRELEVANT OR DUPLICATED ----------------------------------------------------------
# * the same frame stored twice -> exact-copy check
# * irrelevant channels: ch 1 and ch 2 are dominated by impulsive 50 Hz interference -> keep the
#   channel with the strongest band-passed signal
hashes = [hash(np.nan_to_num(data[f]).tobytes()) for f in range(F)]
dup_frames = [f for f in range(F) if hashes[f] in hashes[:f]]
CH = int(np.argmax(np.median(rms, axis=0)))
print("duplicate frames:", dup_frames, "| median band-passed RMS per channel:",
      np.median(rms, axis=0).round(8), "-> channel used:", CH)
cleaning_log.append({"kind": "irrelevant or duplicated", "duplicate_frames": dup_frames, "channel": CH,
                     "action": "drop exact duplicate frames; keep only the strongest channel"})

drop = set(nan_frames) | set(dead_frames) | set(dup_frames)
keep = [f for f in range(F) if f not in drop]
print(f"kept {len(keep)}/{F} frames")

### Build the first batch (a few dozen rows)

5 frames × up to 8 windows of 50 ms. Windows stop when the ring-down falls below 30 % of its peak
(same threshold idea as the authors' `minRelAmplitude`).

In [ ]:
table, X = md.build_table(clean, fs, frames=keep[:5], channel=CH, f_band=BAND,
                          win_s=0.05, ref_s=0.2, n_win=10, min_rel_amp=0.3)
# quality gate: reference fits that explain the segment badly are not trustworthy labels
bad_ref = table["ref_residual_rel"] > 0.5
cleaning_log.append({"kind": "incorrect (labels)", "rows_dropped": int(bad_ref.sum()),
                     "action": "drop rows whose reference-fit residual > 50 % of signal RMS"})
X, table = X[~bad_ref.values], table[~bad_ref].reset_index(drop=True)

table.to_csv(PROC / "windows_table.csv", index=False)
np.save(PROC / "X_windows.npy", X)
print(table.shape, X.shape)
table.head(10)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
for fr, g in table.groupby("frame"):
    ax[0].plot(g["t_start_s"], g["f_ref_hz"], "o-", label=f"frame {fr}")
ax[0].set_xlabel("window start [s]"); ax[0].set_ylabel("f_ref [Hz]"); ax[0].legend(fontsize=7)
ax[1].plot(np.arange(X.shape[1]) / fs * 1e3, X[0], lw=0.8)
ax[1].set_xlabel("t [ms]"); ax[1].set_title("one feature row (window 0)")
plt.tight_layout(); plt.show()
print(table[["f_ref_hz", "A_ref", "snr_db_est", "tau_frame_s", "ref_residual_rel"]].describe().to_string(float_format="%.4g"))

### Irrelevant columns — which columns may be used as features?

Not every column in the table is a legitimate input. IDs (`frame`, `channel`, `t_start_s`) and
constant columns carry no information, and some columns are computed **from the reference fit**, so
they contain the answer in disguise (target leakage). The model input is only `X_windows.npy`.

In [ ]:
const_cols = [c for c in table.columns if table[c].nunique() == 1]
id_cols = ["frame", "channel", "t_start_s"]
leak_cols = ["rel_amplitude", "snr_db_est", "tau_frame_s", "ref_residual_rel"]   # derived from the fit
target_cols = ["f_ref_hz", "A_ref", "phi_ref_rad"]
print("constant columns:", const_cols)
print("corr(t_start_s, f_ref_hz) =", round(table["t_start_s"].corr(table["f_ref_hz"]), 3),
      "-> window position predicts the frequency drift, so it must not be a feature")
cleaning_log.append({"kind": "irrelevant or duplicated (columns)", "constant": const_cols,
                     "ids": id_cols, "target_in_disguise": leak_cols,
                     "action": "not used as features; kept only for filtering/stratifying"})

**First look.** The batch has about 40 windows from 5 frames. The reference frequency rises by about
0.5 Hz as the oscillation decays (τ ≈ 0.41 s, so Q = π f τ ≈ 1000), and the estimated SNR falls from about
28 to 21 dB — the range a short-window estimator will have to handle.

## Step 4 — Metadata

Static facts (source, license, citation, column dictionary) are written by hand in `metadata.yaml`
at the repo root. The cell below adds the facts that are *computed*, so they can never drift from the data.

In [ ]:
meta_path = ROOT / "metadata.yaml"
meta = yaml.safe_load(meta_path.read_text()) if meta_path.exists() else {}
meta["computed"] = {
    "raw_file": RAW.name, "raw_sha256": raw_sha,
    "stored_shape": list(load_info["stored_shape"]), "stored_dtype": load_info["stored_dtype"],
    "fs_hz": float(fs), "n_frames": int(F), "n_channels": int(Cn), "samples_per_frame": int(S),
    "kept_frames": keep, "channel_used": CH,
    "processed_rows": int(len(table)), "window_samples": int(X.shape[1]),
    "cleaning_log": json.loads(json.dumps(cleaning_log, default=str)),
}
meta_path.write_text(yaml.safe_dump(meta, sort_keys=False, allow_unicode=True))
print(meta_path.read_text()[-1500:])

## Step 5 — Version control

Repository: **<link to your repo>** *(or insert a screenshot)*

```
mmr-ml/
├── raw/              # Exp1_MMRS.h5 (git-ignored, 70 MB; download instructions in raw/README.md)
├── processed/        # windows_table.csv, X_windows.npy  (regenerated by this notebook)
├── scripts/          # mmr_data.py  (loading, cleaning, reference fit, table builder)
├── notebooks/        # this notebook
├── metadata.yaml
└── .gitignore
```

## Obstacles and open questions

The first obstacle was that real MMR recordings contain no ground-truth frequency, amplitude or phase,
so the targets are pseudo-labels from a 200 ms reference fit centred on each 50 ms window. Windows near
the frame start, where the reference cannot be centred, had to be dropped, because a shifted reference
gave neighbouring windows identical and biased labels. Second, the reference model itself needed
refinement: a constant-amplitude sinusoid left a 15 % residual that turned out to be the ring-down
itself (τ ≈ 0.41 s, Q ≈ 1000); only after modelling the decay did the residual (4–9 %) and the SNR
estimate (21–28 dB) start to behave like noise. Third, the signal is not a clean sinusoid: its
frequency rises by about 0.5 Hz within 0.35 s as the amplitude decays, odd 50 Hz harmonics remain inside
the 600–2000 Hz band, and two of the three channels carry impulsive 50 Hz interference. Several table
columns turned out to be the target in disguise, because they are computed from the same reference fit.
Finally, the file format is documented mainly through the authors' Julia code, so the axis order, the
units of `/data` (probably volts) and the excitation timing were inferred and remain open questions for
the dataset README. The 70 MB raw file is kept out of Git and identified by its SHA-256 hash. This sensor
(≈810 Hz, 1 mm rotor) also differs from my own design (≈570 Hz), so the data test the method rather than
my design's numbers.